In [ ]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

In [2]:
import os
import json
import pandas as pd
import time
import random
from datetime import datetime, timedelta
import faiss
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer

c:\Users\Danh\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
def _get_store_paths(store_name: str):
    """Tạo đường dẫn file động cho một kho tri thức cụ thể."""
    base_dir = r"D:\finalproject\KLTN\Backend\data\vector_store"
    index_path = os.path.join(base_dir, f"faiss_index_{store_name}.bin")
    docs_path = os.path.join(base_dir, f"documents_{store_name}.json")
    return index_path, docs_path

_stores = {}

def get_store(store_name: str):
    """
    Lấy một kho tri thức cụ thể. Tải từ cache nếu có, nếu không thì xây dựng mới.
    """
    if store_name in _stores:
        return _stores[store_name]

    index_path, docs_path = _get_store_paths(store_name)

    if os.path.exists(index_path) and os.path.exists(docs_path):
        try:
            print(f"Đang tải kho '{store_name}' từ cache...")
            index = faiss.read_index(index_path)
            with open(docs_path, 'r', encoding='utf-8') as f:
                documents = json.load(f)
            print(f"Tải thành công kho '{store_name}' với {index.ntotal} vector.")
            
            store_instance = {"index": index, "documents": documents}
            _stores[store_name] = store_instance
            return store_instance
        except Exception as e:
            print(f"Lỗi khi tải kho '{store_name}' từ cache: {e}. Sẽ xây dựng lại.")

def retrieve(store_name: str, query: str, k: int = 5) -> str:
    """Thực hiện truy vấn trên một kho tri thức chuyên biệt."""
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
    store = get_store(store_name)
    if not store or store.get("index") is None:
        print(f"Truy vấn thất bại: Kho tri thức '{store_name}' chưa được khởi tạo.")
        return f"Lỗi: Cơ sở tri thức '{store_name}' không khả dụng."

    index = store["index"]
    documents = store["documents"]
    
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )
    
    try:
        _, indices = index.search(np.array(query_embedding, dtype=np.float32), k)
        retrieved_docs = [documents[i] for i in indices[0]]
        context = "\n---\n".join([doc['content'] for doc in retrieved_docs])
        
        print(f"Đã truy xuất {len(retrieved_docs)} đoạn văn bản từ kho '{store_name}' cho câu hỏi: '{query[:50]}...'")
        return context
    except Exception as e:
        print(f"Lỗi trong quá trình truy xuất từ kho '{store_name}': {e}")
        return "Lỗi: Đã xảy ra sự cố khi tìm kiếm thông tin."

In [24]:
import json
from datetime import datetime, timezone
from typing import Dict, List, Optional

# --- 1. Hàm Xây Dựng Prompt (Đã chuẩn hóa input) ---
def _build_treatment_prompt_final(
    retrieved_context: str,
    farmer_info: Dict,
    daily_summary_3d: Dict,
    iot_data: Dict,
    disease_name_vn: str,
    days_since_planting: int
) -> str:
    try:
        last_date_str = list(daily_summary_3d.keys())[-1] # vd: "3/11/2025"
        current_simulated_time = datetime.strptime(last_date_str, "%d/%m/%Y").isoformat()
    except:
        current_simulated_time = datetime.now().isoformat()
    farmer_json = json.dumps(farmer_info, ensure_ascii=False, indent=2)
    weather_json = json.dumps(daily_summary_3d, ensure_ascii=False, indent=2)
    iot_json = json.dumps(iot_data, ensure_ascii=False, indent=2)

    # Schema + example trả về JSON chính xác (ISO date + session restricted)
    prompt = f"""
        Bạn là chuyên gia nông nghiệp AI. Thời gian: {current_simulated_time}.
        Nhiệm vụ: dựa trên dữ liệu, trả về duy nhất 1 JSON hợp lệ (no extra text).
        Điều bắt buộc:
        - Trường `optimal_spray_day.date` phải là ISO 8601 UTC (YYYY-MM-DDTHH:MM:SSZ).
        - Trường `optimal_spray_day.session` chỉ được là một trong: "Sáng", "Chiều", "Sáng sớm", "Chiều muộn", "Tối".
        - Trường `treatment_plan.drug_info.hoạt_chất` bắt buộc phải không rỗng nếu thuốc được khuyến nghị.
        - Trả về JSON duy nhất, không giải thích.
        - Trong phần "risk_assessment" và "weather_summary" PHẢI trích dẫn ít nhất 6 con số cụ thể từ dữ liệu thời tiết 3 ngày và IoT (nhiệt độ, độ ẩm, lượng mưa, UV, ánh sáng, v.v.). Nếu không đủ 6 số → is_action_needed = false.
        - Trường "hoạt_chất" CHỈ được lấy từ Context đã cung cấp. Nếu Context không có hoạt chất phù hợp → để trống và ghi trong main_message: "Cần tham khảo thêm chuyên gia địa phương".

        Dữ liệu:
        - Bệnh: {disease_name_vn}, {days_since_planting} NSS.
        - Context:
        {retrieved_context}

        Thông tin nông hộ:
        {farmer_json}

        Thời tiết 3 ngày:
        {weather_json}

        IoT hiện tại:
        {iot_json}

        === OUTPUT FORMAT (literal JSON schema) ===
        {{
        "is_action_needed": true|false,
        "analysis": {{
            "risk_assessment": "text",
            "weather_summary": "text"
        }},
        "treatment_plan": {{
            "is_actionable": true|false,
            "main_message": "text",
            "optimal_spray_day": {{
            "date": "2025-11-03T06:00:00Z",
            "session": "Sáng sớm",
            "reason": "text (brief)"
            }},
            "drug_info": {{
            "sản_phẩm_tham_khảo": "text",
            "hoạt_chất": "Tricyclazole",
            "liều_lượng": "text with numbers and units"
            }},
            "additional_actions": ["text", "..."]
        }},
        "fertilizer_advice": {{
            "recommendation": "text",
            "reason": "text"
        }},
        "prognosis": "text"
        }}

        === EXAMPLE OUTPUT (one-line JSON only) ===
        {{"is_action_needed":true, "analysis":{{"risk_assessment":"...","weather_summary":"..."}}, "treatment_plan":{{"is_actionable":true,"main_message":"...","optimal_spray_day":{{"date":"2025-11-03T06:30:00Z","session":"Sáng sớm","reason":"..."}}, "drug_info":{{"sản_phẩm_tham_khảo":"Fujione 40EC","hoạt_chất":"Tricyclazole","liều_lượng":"1.0 lít/ha"}},"additional_actions":["..."]}},"fertilizer_advice":{{"recommendation":"...","reason":"..."}},"prognosis":"..."}}
    """
    return prompt

# --- 2. Hàm Tạo Kế Hoạch (Agent) ---
def create_treatment_plan_from_sample(sample_item: Dict, client_instance):
    """
    Hàm này nhận một item trong sample_farmer_data, 
    tự động tách expect_plan ra, và dùng phần còn lại để tạo kế hoạch.
    """
    # 1. Tách dữ liệu
    # Sao chép để không ảnh hưởng dữ liệu gốc
    input_data = sample_item.copy()
    
    # Loại bỏ expect_plan khỏi đầu vào cho AI (để tránh AI "nhìn bài")
    if 'expect_plan' in input_data:
        del input_data['expect_plan']
        
    farmer_id = input_data.get('farmer_id')
    disease_name_raw = input_data['iot_data'].get('detected_disease_name', 'unknown')
    days_since_planting = input_data.get('days_since_planting', 40)
    
    # Map tên bệnh
    disease_map = {"blast": "Đạo ôn", "brown_spot": "Đốm nâu", "bacterial_leaf_blight": "Cháy bìa lá"}
    disease_name_vn = disease_map.get(disease_name_raw, "Không xác định")

    # Giả lập Retrieve Context (Trong thực tế sẽ gọi Vector DB)
    # Thay đoạn query giả lập cũ bằng cái này:
    query = (
        f"bệnh {disease_name_vn} "
        f"giai đoạn {days_since_planting} ngày sau sạ "
        f"hoạt chất đặc trị khuyến cáo thuốc bảo vệ thực vật "
    )
    retrieved_context = retrieve(disease_name_raw, query, k=20)  

    # 2. Build Prompt
    # Lấy thông tin nông hộ (trừ các cục data to)
    farmer_info_clean = {k: v for k, v in input_data.items() if k not in ['summary_3d', 'iot_data']}
    
    prompt = _build_treatment_prompt_final(
        retrieved_context=retrieved_context,
        farmer_info=farmer_info_clean,
        daily_summary_3d=input_data.get('summary_3d', {}),
        iot_data=input_data.get('iot_data', {}),
        disease_name_vn=disease_name_vn,
        days_since_planting=days_since_planting
    )

    # 3. Gọi LLM
    if not client_instance:
        return {"error": "Client chưa sẵn sàng"}
        
    try:
        response = client_instance.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.3
        )
        plan_content = response.choices[0].message.content
        return json.loads(plan_content)
    except Exception as e:
        print(f"Error calling LLM: {e}")
        return {"error": str(e)}

In [25]:
import json
import re
from typing import Dict, List
from datetime import datetime
from dateutil.parser import parse

# -----------------------------
def _normalize_session(session_str: str) -> str:
    """
    Chuẩn hóa buổi phun: 'Sáng sớm 6h-9h30' -> 'sáng', 'Chiều muộn' -> 'chiều'
    """
    s = session_str.lower().replace(' ', '').replace('–', '-')
    if 'sáng' in s:
        return 'sáng'
    if 'chiều' in s or 'tối' in s:
        return 'chiều' # Gộp tối vào chiều (chiều muộn) vì đều liên quan đến độ ẩm, tránh nắng gắt
    return 'khác'

# -----------------------------
# HÀM PHỤ: ĐẾM SỐ LIỆU / ĐƠN VỊ (Giữ nguyên)
# -----------------------------
def _count_data_mentions(text: str) -> int:
    # Thêm nhiều pattern hơn
    patterns = [
        r'\d+[.,]?\d*\s*(°C|°c|%|cm|m/s|lux|ha|kg|lít|ml|g)',
        r'\d+\s*[-–]\s*\d+\s*(°C|%|cm|m/s|lux)',
        r'(?:sáng|chiều|trưa)\s*\d+h',
        r'\d+\s*ngày\s*sau\s*sạ',
    ]
    return sum(len(re.findall(p, text, re.I)) for p in patterns)

def _good_risk_assessment(text: str) -> bool:
    text = text.lower()
    if len(text) < 40:
        return False
    strong_keywords = [
        "nguy cơ cao", "nghiêm trọng", "khẩn cấp", "cực kỳ nguy hiểm", "bùng phát mạnh",
        "mức độ cao", "rủi ro cao", "cảnh báo đỏ", "cần xử lý ngay", "đang lây lan nhanh"
    ]
    mild_keywords = ["thấp", "trung bình", "ổn định", "chưa nghiêm trọng"]
    # Ưu tiên từ mạnh, nếu không có từ mạnh nhưng có từ nhẹ + giải thích thì vẫn chấp nhận
    if any(k in text for k in strong_keywords):
        return True
    if any(k in text for k in mild_keywords) and len(text) > 80:
        return True
    return False


# -----------------------------
# HÀM CHÍNH: ĐÁNH GIÁ KẾ HOẠCH
# -----------------------------
def evaluate_plan_vs_expect(actual_plan: Dict, expected_plan: Dict, latency_s: float) -> Dict:
    metrics = {
        "action_accuracy": 0,
        "grounding_mentions": 0,
        "grounding_enough": 0,   
        "risk_assessment_accuracy": 0,
        "spray_date_accuracy": 0,
        "spray_session_accuracy": 0,
        "dose_quality_accuracy": 0,
        "latency_s": latency_s,
        "details": []
    }

    if actual_plan is None or "error" in actual_plan:
        metrics["details"].append("❌ Kế hoạch lỗi, không đánh giá được.")
        return metrics

    # =============================
    # 1) ACTION DECISION
    # =============================
    if actual_plan.get("is_action_needed") == expected_plan.get("is_action_needed"):
        metrics["action_accuracy"] = 1
        metrics["details"].append("✅ Action Decision chính xác.")
    else:
        metrics["details"].append("❌ Sai action decision.")

    # =============================
    # 2) GROUNDING QUALITY
    # =============================
    analysis = actual_plan.get("analysis", {})
    full_text = analysis.get("risk_assessment", "") + " " + analysis.get("weather_summary", "")

    count = _count_data_mentions(full_text)
    metrics["grounding_mentions"] = count
    metrics["grounding_enough"] = 1 if count >= 8 else 0

    metrics["details"].append(f"📊 Grounding: {count} số liệu (>=8 là đạt).")

    # =============================
    # 3) RISK ASSESSMENT
    # =============================
    risk_text = analysis.get("risk_assessment", "").lower()
    if _good_risk_assessment(risk_text):
        metrics["risk_assessment_accuracy"] = 1
        metrics["details"].append("🛡️ Risk Assessment đạt yêu cầu.")
    else:
        metrics["details"].append("⚠️ Risk Assessment chưa đạt.")

    # =============================
    # 4) OPTIMAL SPRAY DAY
    # =============================
    try:
        act_day = actual_plan["treatment_plan"]["optimal_spray_day"]
        exp_day = expected_plan["treatment_plan"]["optimal_spray_day"]

        act_date_str = act_day["date"]
        act_date = parse(act_date_str.split("T")[0]).date()
        exp_date = parse(exp_day["date"].split("T")[0]).date()
        if act_date == exp_date:
            metrics["spray_date_accuracy"] = 1

        act_sess = _normalize_session(act_day.get("session", ""))
        exp_sess = _normalize_session(exp_day.get("session", ""))

        if act_sess == exp_sess:
            metrics["spray_session_accuracy"] = 1

        metrics["details"].append(f"🌤️ Spray day AI: {act_date} ({act_sess}) – EXP: {exp_date} ({exp_sess})")

    except Exception:
        metrics["details"].append("❌ Không parse được ngày/buổi phun.")

    # =============================
    # 5) DOSE QUALITY ACCURACY
    # =============================
    dose = actual_plan.get("treatment_plan", {}).get("drug_info", {}).get("liều_lượng", "").lower()
    pattern = r'\d+(?:\.\d+)?\s*[-–]?\s*\d*\s*(?:ml|l|g|kg|lít|gam|gram|chai|gói)\s*/\s*(?:ha|bình|công|sào)'
    
    if re.search(pattern, dose, flags=re.IGNORECASE):
        metrics["dose_quality_accuracy"] = 1
        metrics["details"].append(f"🧪 Liều lượng đúng chuẩn: {dose}")
    else:
        metrics["details"].append(f"⚠️ Liều lượng không đúng định dạng: {dose}")
    return metrics

In [26]:
sample_farmer_data = [
    {
        "farmer_id": "FARMER001",
        "full_name": "Nguyễn Văn An",
        "farm_name": "Ruộng An Phát",
        "area_ha": 2.5,
        "planting_date": "2025-09-20",
        "days_since_planting": 44,                   
        "location": {
            "province": "An Giang"
        },
        "iot_data": { 
            "temperature": 29.5,
            "humidity": 83,
            "soil_moisture": 65,
            "soil_ph": 5.8,
            "water_level": 10,
            "lux": 48200,
            "wind": 2.8,
            "wind_avg": 2.4,
            "detected_disease_name": "blast",
            "disease_confidence": 0.97
        },
        "summary_3d": {  
            "1/11/2025": {
                "temperature": 28.6, "humidity": 91, "soil_moisture": 72, "soil_ph": 5.9,
                "water_level": 15, "lux": 35800, "wind": 1.3, "wind_avg": 1.1
            },
            "2/11/2025": {
                "temperature": 28.9, "humidity": 88, "soil_moisture": 69, "soil_ph": 5.8,
                "water_level": 13, "lux": 41200, "wind": 1.7, "wind_avg": 1.5
            },
            "3/11/2025": {  
                "temperature": 29.5, "humidity": 83, "soil_moisture": 65, "soil_ph": 5.8,
                "water_level": 10, "lux": 48200, "wind": 2.8, "wind_avg": 2.4
            }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "KHẨN CẤP CAO NHẤT",  
            "analysis": {
                "risk_assessment": "Bệnh đạo ôn lá (đạo ôn) đang ở mức CỰC KỲ NGUY HIỂM: độ tin cậy phát hiện tăng nhanh từ 89% → 97% chỉ trong 3 ngày. Ngày 03/11/2025 ghi nhận điều kiện thời tiết HOÀN HẠO cho bệnh phát triển mạnh (nhiệt độ cao, ánh sáng mạnh) nhưng LẠI LÀ ĐIỀU KIỆN LÝ TƯỞNG ĐỂ PHUN THUỐC (độ ẩm giảm mạnh xuống 83%, gió tốt 2.8 m/s, nắng mạnh → thuốc khô nhanh, bám dính cực tốt).",
                "weather_summary": "Hôm nay 03/11/2025 là NGÀY VÀNG DUY NHẤT trong ít nhất 3–5 ngày tới để xử lý đạo ôn đạt hiệu quả tối đa. Nếu bỏ lỡ hôm nay, bệnh sẽ bùng phát không kiểm soát được do điều kiện đang cực kỳ thuận lợi cho nấm."
            },
            "treatment_plan": {
                "is_actionable": True,
                "main_message": "PHẢI PHUN THUỐC TRỪ ĐẠO ÔN KHẨN CẤP NGAY TRONG SÁNG HÔM NAY 03/11/2025 (trước 10h sáng). KHÔNG ĐƯỢC CHẦN CHỪ THÊM DÙ CHỈ 1 NGÀY!",
                "optimal_spray_day": {
                    "date": "2025-11-03",
                    "session": "Sáng sớm 6h00 – 9h30",
                    "reason": "Nhiệt độ 29.5°C, độ ẩm chỉ 83% (thấp nhất 3 ngày), nắng mạnh 48.200 lux, gió 2.4–2.8 m/s → thuốc khô nhanh trong 30–60 phút, bám dính cực tốt, hiệu lực trừ bệnh đạt >95%. Đây là cửa sổ thời tiết HIẾM CÓ – không thể tốt hơn!"
                },
                "drug_info": {
                    "sản_phẩm_khuyến_cáo": "Fujione 40EC (Syngenta) hoặc Beam 75WP (Bayer) hoặc Nativo 75WG",
                    "hoạt_chất": "Tricyclazole (ưu tiên) hoặc Tricyclazole + Kasugamycin hoặc Tebuconazole + Trifloxystrobin",
                    "liều_lượng": "1.0 lít/ha Fujione hoặc 0.6–0.8 kg/ha Beam/Nativo",
                    "tổng_cho_2.5ha": "2.5 lít Fujione hoặc 1.5–2.0 kg Beam/Nativo",
                    "pha_bình_25l": "30–35ml Fujione hoặc 18–20g Beam/Nativo cho bình 25 lít",
                    "cách_phun": "Phun ướt đều 2 mặt lá, tập trung lá đòng, cổ bông và phần ngọn. Dùng bình phun sương mịn, áp suất cao. Phun xong giữ mực nước ruộng 3–5cm."
                },
                "additional_actions_right_now": [
                    "Phun NGAY trong sáng nay 03/11/2025",
                    "Mang đầy đủ bảo hộ: khẩu trang, kính, quần áo dài, ủng",
                    "Sau phun 6h không để mưa (hôm nay không mưa → an toàn)",
                    "Giữ nước ruộng 3–5cm trong 5–7 ngày tới",
                    "Bón bổ sung ngay 15kg KCl + 10kg urê/ha trong 1–2 ngày tới",
                    "Theo dõi lại sau 5 ngày, sẵn sàng phun nhắc lại nếu cần"
                ]
            },
            "fertilizer_advice": {
                "immediate": "Bón ngay 15kg KCl/ha + 10kg Đạm urê/ha trong vòng 48h tới",
                "reason": "Kali giúp tăng sức đề kháng, làm dày biểu bì lá, giảm lấp lánh bệnh. Đạm giúp cây hồi phục nhanh sau khi bị bệnh tấn công."
            },
            "prognosis": {
                "if_act_today": "Nếu phun đúng và đủ liều trong sáng nay → 90–95% khả năng khống chế hoàn toàn đạo ôn trong 7 ngày, tỷ lệ cháy lá <3%, năng suất dự kiến vẫn giữ được 7.5–8.0 tấn/ha.",
                "if_delay": "Nếu trì hoãn dù chỉ 1–2 ngày → bệnh sẽ lan toàn ruộng, cháy lá >30%, mất trắng cổ bông, năng suất giảm còn <4 tấn/ha."
            }
        }
    },
    {
        "farmer_id": "FARMER002",
        "full_name": "Trần Thị Bé Hai",
        "farm_name": "Tân Phát Lộc",
        "area_ha": 1.8,
        "planting_date": "2025-09-10",
        "days_since_planting": 54,
        "location": {
            "province": "Sóc Trăng"
        },
        "iot_data": {
            "temperature": 27.8,
            "humidity": 88,
            "soil_moisture": 75,
            "soil_ph": 5.4,
            "water_level": 18,
            "lux": 32000,
            "wind": 1.5,
            "wind_avg": 1.2,
            "detected_disease_name": "brown_spot",
            "disease_confidence": 0.72
        },
        "summary_3d": {
            "2/11/2025": {
                "temperature": 27.0, "humidity": 93, "soil_moisture": 78, "soil_ph": 5.5,
                "water_level": 20, "lux": 28000, "wind": 0.9, "wind_avg": 0.7
            },
            "3/11/2025": {
                "temperature": 27.5, "humidity": 90, "soil_moisture": 76, "soil_ph": 5.4,
                "water_level": 19, "lux": 30000, "wind": 1.1, "wind_avg": 0.8
            },
            "4/11/2025": {
                "temperature": 27.8, "humidity": 88, "soil_moisture": 75, "soil_ph": 5.4,
                "water_level": 18, "lux": 32000, "wind": 1.5, "wind_avg": 1.2
            }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "CẢNH BÁO",
            "analysis": {
                "risk_assessment": "Bệnh đốm nâu (brown spot) đang chớm phát triển do thiếu dinh dưỡng Kali/Lân (pH đất 5.4 khá chua). Độ tin cậy 72%. Mức độ nguy hiểm thấp hơn đạo ôn nhưng nếu không xử lý sẽ làm giảm diện tích quang hợp, cây yếu, dễ nhiễm các bệnh khác.",
                "weather_summary": "Thời tiết hiện tại vẫn ẩm, mát. Ưu tiên phun thuốc trong điều kiện tạnh ráo, có nắng nhẹ. Giải pháp chính là điều chỉnh dinh dưỡng."
            },
            "treatment_plan": {
                "is_actionable": True,
                "main_message": "PHUN THUỐC TRỪ NẤM PHỔ RỘNG VÀ BÓN BỔ SUNG LÂN/KALI KHẨN CẤP.",
                "optimal_spray_day": {
                    "date": "2025-11-04",
                    "session": "Sáng",
                    "reason": "Dự báo ngày mai có nắng tốt hơn (lux tăng), độ ẩm giảm nhẹ. Phun buổi chiều giúp thuốc bám dính lâu hơn qua đêm trước khi bón phân."
                },
                "drug_info": {
                    "sản_phẩm_khuyến_cáo": "Score 250 EC (Syngenta) hoặc Amistar Top 325 SC",
                    "hoạt_chất": "Difenoconazole hoặc Azoxystrobin + Difenoconazole",
                    "liều_lượng": "0.3 lít/ha Score hoặc 0.5 lít/ha Amistar Top",
                    "tổng_cho_1.8ha": "0.54 lít Score hoặc 0.9 lít Amistar Top",
                    "pha_bình_25l": "10–12ml Score hoặc 15–20ml Amistar Top cho bình 25 lít",
                    "cách_phun": "Phun ướt đều 2 mặt lá. KHÔNG trộn với phân bón lá có đạm cao."
                },
                "additional_actions_right_now": [
                    "Phun thuốc vào sáng 03/11/2025",
                    "Bón ngay 20kg Super Lân + 15kg KCl/ha để cải tạo dinh dưỡng và tăng sức đề kháng.",
                    "Tháo bớt nước ruộng về mức 5-7cm để giảm độ ẩm gốc cây."
                ]
            },
            "fertilizer_advice": {
                "immediate": "Bón ngay 20kg Lân + 15kg Kali/ha trong vòng 48h tới",
                "reason": "Bón Lân và Kali là giải pháp gốc cho bệnh đốm nâu do bệnh thường liên quan đến thiếu dinh dưỡng."
            },
            "prognosis": {
                "if_act_today": "Nếu xử lý kết hợp thuốc và phân bón trong 48h → 85% khả năng kiểm soát bệnh, cây phục hồi nhanh, năng suất ổn định.",
                "if_delay": "Nếu trì hoãn → bệnh lây lan mạnh, giảm khả năng quang hợp, làm hạt lép lửng."
            }
        }
    },
    {
        "farmer_id": "FARMER003",
        "full_name": "Phạm Văn Thiện",
        "farm_name": "Ruộng Lúa Vàng",
        "area_ha": 4.0,
        "planting_date": "2025-10-01",
        "days_since_planting": 33,
        "location": {
            "province": "Long An"
        },
        "iot_data": {
            "temperature": 26.5,
            "humidity": 95,
            "soil_moisture": 85,
            "soil_ph": 6.2,
            "water_level": 25,
            "lux": 15000,
            "wind": 3.8,
            "wind_avg": 3.5,
            "detected_disease_name": "bacterial_leaf_blight",
            "disease_confidence": 0.88
        },
        "summary_3d": {
            "1/11/2025": {
                "temperature": 27.0, "humidity": 92, "soil_moisture": 80, "soil_ph": 6.1,
                "water_level": 22, "lux": 25000, "wind": 1.5, "wind_avg": 1.2
            },
            "2/11/2025": {
                "temperature": 26.8, "humidity": 94, "soil_moisture": 83, "soil_ph": 6.2,
                "water_level": 23, "lux": 18000, "wind": 2.5, "wind_avg": 2.2
            },
            "3/11/2025": {
                "temperature": 26.5, "humidity": 95, "soil_moisture": 85, "soil_ph": 6.2,
                "water_level": 25, "lux": 15000, "wind": 3.8, "wind_avg": 3.5
            }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "KHẨN CẤP",
            "analysis": {
                "risk_assessment": "Bệnh bạc lá (vi khuẩn) đang lây lan nhanh (88% tin cậy). Điều kiện thời tiết **MƯA LỚN, ẨM ĐỘ VÀ GIÓ CỰC CAO** trong 3 ngày qua là hoàn hảo cho vi khuẩn lây lan theo vết thương cơ giới và nước. Gió mạnh hôm nay (3.8 m/s) làm vi khuẩn càng dễ lây lan.",
                "weather_summary": "Phải xử lý kháng sinh ngay lập tức kết hợp tháo nước ruộng để hạn chế lây lan qua nguồn nước. Cần theo dõi dự báo mưa."
            },
            "treatment_plan": {
                "is_actionable": True,
                "main_message": "PHUN KHÁNG SINH ĐẶC TRỊ BẠC LÁ KHẨN CẤP VÀ THÁO NƯỚC RUỘNG NGAY LẬP TỨC.",
                "optimal_spray_day": {
                    "date": "2025-11-03",
                    "session": "Buổi sáng/chiều",
                    "reason": "Cần tranh thủ phun ngay khi tạnh mưa. Buổi trưa/chiều nhiệt độ có thể cao hơn, giúp thuốc khô và hấp thụ nhanh hơn."
                },
                "drug_info": {
                    "sản_phẩm_khuyến_cáo": "Starner 20 WP (Oxolinic acid) hoặc Kasumin 2L (Kasugamycin)",
                    "hoạt_chất": "Oxolinic acid (ưu tiên) hoặc Kasugamycin",
                    "liều_lượng": "0.8 kg/ha Starner hoặc 1.0 lít/ha Kasumin",
                    "tổng_cho_4.0ha": "3.2 kg Starner hoặc 4.0 lít Kasumin",
                    "pha_bình_25l": "25–30g Starner hoặc 30–35ml Kasumin cho bình 25 lít",
                    "cách_phun": "Phun ướt đều toàn bộ lá. Không phun lúc trời sắp mưa. NÊN PHUN NHẮC LẠI SAU 5 NGÀY."
                },
                "additional_actions_right_now": [
                    "Tháo nước ruộng triệt để trong 24h để hạn chế lây lan. CHỈ GIỮ 1-2CM NƯỚC.",
                    "Ngưng bón phân Đạm trong 1 tuần.",
                    "Phun ngay trong hôm nay 03/11/2025, theo dõi và phun nhắc lại sau 5 ngày."
                ]
            },
            "fertilizer_advice": {
                "immediate": "NGƯNG BÓN ĐẠM trong 1 tuần. Chỉ bón 10kg KCl/ha (Kali) sau khi phun thuốc 2 ngày.",
                "reason": "Đạm kích thích cây non, làm vi khuẩn dễ tấn công hơn. Kali tăng sức đề kháng."
            },
            "prognosis": {
                "if_act_today": "Nếu phun kháng sinh và tháo nước kịp thời → 70% khả năng kiểm soát trong 10 ngày, tránh mất năng suất đáng kể.",
                "if_delay": "Nếu trì hoãn → bệnh lây lan toàn ruộng, tỷ lệ cháy lá lên đến 50%, mất trắng."
            }
        }
    },
    {
        "farmer_id": "FARMER005",
        "full_name": "Nguyễn Văn Lượm",
        "farm_name": "Ruộng Miền Tây",
        "area_ha": 1.5,
        "planting_date": "2025-08-15",
        "days_since_planting": 80,
        "location": {
            "province": "Bến Tre"
        },
        "iot_data": {
            "temperature": 26.0,
            "humidity": 90,
            "soil_moisture": 78,
            "soil_ph": 6.0,
            "water_level": 12,
            "lux": 25000,
            "wind": 1.0,
            "wind_avg": 0.8,
            "detected_disease_name": "blast",
            "disease_confidence": 0.55
        },
        "summary_3d": {
            "1/11/2025": {
                "temperature": 26.5, "humidity": 88, "soil_moisture": 75, "soil_ph": 6.0,
                "water_level": 10, "lux": 30000, "wind": 1.5, "wind_avg": 1.2
            },
            "2/11/2025": {
                "temperature": 26.2, "humidity": 89, "soil_moisture": 77, "soil_ph": 6.0,
                "water_level": 11, "lux": 28000, "wind": 1.2, "wind_avg": 1.0
            },
            "3/11/2025": {
                "temperature": 26.0, "humidity": 90, "soil_moisture": 78, "soil_ph": 6.0,
                "water_level": 12, "lux": 25000, "wind": 1.0, "wind_avg": 0.8
            }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "PHÒNG NGỪA",
            "analysis": {
                "risk_assessment": "Lúa đang ở giai đoạn trổ bông. Điều kiện thời tiết **ẩm độ cao (90%), nhiệt độ mát** là nguy cơ CỰC CAO cho bệnh Đạo Ôn Cổ Bông. Mặc dù độ tin cậy phát hiện chỉ 55% (chớm), NHƯNG PHẢI PHÒNG NGỪA VÌ LÚA ĐÃ TRỔ.",
                "weather_summary": "Phòng ngừa là bắt buộc. Phun ngay khi lúa trổ lác đác 5-10% và phun nhắc lại khi lúa trổ đều 80-90%."
            },
            "treatment_plan": {
                "is_actionable": True,
                "main_message": "PHUN PHÒNG NGỪA ĐẠO ÔN CỔ BÔNG NGAY LẬP TỨC (đợt 1) VÀ LÊN KẾ HOẠCH PHUN NHẮC LẠI.",
                "optimal_spray_day": {
                    "date": "2025-11-03",
                    "session": "Buổi sáng/chiều (Tùy điều kiện tạnh ráo)",
                    "reason": "Phun ngay khi lúa trổ lác đác (5-10% diện tích). Tránh phun lúc sương muối hoặc quá ẩm."
                },
                "drug_info": {
                    "sản_phẩm_khuyến_cáo": "Beam 75WP (Bayer) hoặc Nativo 75WG",
                    "hoạt_chất": "Tricyclazole (ưu tiên) hoặc Tebuconazole + Trifloxystrobin",
                    "liều_lượng": "0.8 kg/ha Beam hoặc 0.8 kg/ha Nativo",
                    "tổng_cho_1.5ha": "1.2 kg Beam hoặc 1.2 kg Nativo",
                    "pha_bình_25l": "20–25g Beam/Nativo cho bình 25 lít",
                    "cách_phun": "Phun ướt đều cổ bông và lá đòng. Cần phun 2 đợt: đợt 1 khi trổ lác đác, đợt 2 khi trổ đều (sau 5-7 ngày)."
                },
                "additional_actions_right_now": [
                    "Phun phòng ngừa đợt 1 ngay hôm nay 03/11/2025.",
                    "Lên lịch phun nhắc lại đợt 2 sau 5–7 ngày.",
                    "Bón nhẹ 10kg Kali/ha để tăng sức chống chịu cổ bông."
                ]
            },
            "fertilizer_advice": {
                "immediate": "Bón 10kg KCl/ha (Kali) để tăng sức chống chịu bệnh cổ bông.",
                "reason": "Kali rất quan trọng trong giai đoạn trổ để ngăn ngừa nấm tấn công cổ bông."
            },
            "prognosis": {
                "if_act_today": "Nếu phun phòng ngừa 2 đợt đúng lúc → 98% bảo vệ cổ bông hoàn toàn, năng suất tối đa.",
                "if_delay": "Nếu trì hoãn → bệnh cổ bông bùng phát nhanh, tỷ lệ lép hạt, gãy cổ bông có thể lên đến 40–50%."
            }
        }
    },
    {
        "farmer_id": "FARMER012",
        "full_name": "Trần Thị Mười",
        "farm_name": "Ruộng Đồng Tháp Mười",
        "area_ha": 2.0,
        "planting_date": "2025-08-10",
        "days_since_planting": 85,
        "location": { "province": "Đồng Tháp" },
        "iot_data": {
            "temperature": 25.8,
            "humidity": 92,
            "soil_moisture": 80,
            "soil_ph": 5.8,
            "water_level": 15,
            "lux": 22000,
            "wind": 0.5,
            "wind_avg": 0.6,
            "detected_disease_name": "blast",
            "disease_confidence": 0.88
        },
        "summary_3d": {
            "8/11/2025": { "temperature": 25.5, "humidity": 94, "soil_moisture": 82, "water_level": 16, "lux": 18000, "wind": 0.3 },
            "9/11/2025": { "temperature": 25.7, "humidity": 93, "soil_moisture": 81, "water_level": 15, "lux": 21000, "wind": 0.4 },
            "10/11/2025": { "temperature": 25.8, "humidity": 92, "soil_moisture": 80, "water_level": 15, "lux": 22000, "wind": 0.5 }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "KHẨN CẤP",
            "analysis": {
            "risk_assessment": "Lúa đang trổ đều (80–90%), phát hiện Đạo ôn cổ bông với độ tin cậy 88%. Độ ẩm liên tục >90%, sương mù buổi sáng → điều kiện lý tưởng cho nấm Pyricularia phát triển mạnh.",
            "weather_summary": "3 ngày tới vẫn ẩm cao, không có ngày nắng ráo hoàn toàn."
            },
            "treatment_plan": {
            "is_actionable": True,
            "main_message": "PHUN KHẨN CẤP ĐỢT 1 NGAY NGÀY 10/11/2025 VÀ NHẮC LẠI SAU 5 NGÀY",
            "optimal_spray_day": {
                "date": "2025-11-10",
                "session": "Buổi chiều sau 15h (khi sương đã tan)",
                "reason": "Ngày duy nhất có gió nhẹ và độ ẩm giảm nhẹ vào chiều tối trong 5 ngày tới."
            },
            "drug_info": {
                "sản_phẩm_khuyến_cáo": "Beam 75WP hoặc Fuji-One 40EC",
                "hoạt_chất": "Tricyclazole 75% hoặc Isoprothiolane 40%",
                "liều_lượng": "0.8–1.0 kg/ha Beam hoặc 1.0–1.2 lít/ha Fuji-One",
                "tổng_cho_2.0ha": "1.6–2.0 kg Beam hoặc 2.0–2.4 lít Fuji-One",
                "pha_bình_25l": "25–30g Beam hoặc 30–35ml Fuji-One",
                "cách_phun": "Phun tập trung vào cổ bông và lá đòng, phun ướt đều 2 mặt lá. Phun đợt 2 sau 5 ngày."
            },
            "additional_actions_right_now": [
                "Phun khẩn cấp ngay 10/11/2025",
                "Tháo cạn nước ruộng 1–2 ngày sau phun để giảm ẩm",
                "Bón 15kg KCl/ha tăng sức chống chịu"
            ]
            }
        }
    },
    {
        "farmer_id": "FARMER028",
        "full_name": "Lê Văn Hữu Phước",
        "farm_name": "Ruộng An Giang",
        "area_ha": 1.8,
        "planting_date": "2025-08-20",
        "days_since_planting": 82,
        "location": { "province": "An Giang" },
        "iot_data": {
            "temperature": 27.5,
            "humidity": 88,
            "soil_moisture": 85,
            "soil_ph": 5.5,
            "water_level": 18,
            "lux": 35000,
            "wind": 3.5,
            "wind_avg": 3.2,
            "detected_disease_name": "bacterial_leaf_blight",
            "disease_confidence": 0.82
        },
        "summary_3d": {
            "15/11/2025": { "temperature": 28.0, "humidity": 85, "soil_moisture": 87, "water_level": 20, "wind": 4.0 },
            "16/11/2025": { "temperature": 27.8, "humidity": 86, "soil_moisture": 86, "water_level": 19, "wind": 3.8 },
            "17/11/2025": { "temperature": 27.5, "humidity": 88, "soil_moisture": 85, "water_level": 18, "wind": 3.5 }
        },
        "expect_plan": {
            "is_action_needed": True,
            "treatment_plan": {
            "is_actionable": True,
            "optimal_spray_day": {
                "date": "2025-11-17",  
                "session": "Buổi sáng sớm 6–9h",
                "reason": "Gió mạnh giúp thuốc đồng bám đều, khô nhanh, hạn chế rửa trôi."
            },
            "drug_info": {
                "sản_phẩm_khuyến_cáo": "Kasumin 2L hoặc Bactericide 20WP + Đồng oxyclorua",
                "hoạt_chất": "Kasugamycin hoặc Copper hydroxide + Kasugamycin",
                "liều_lượng": "1.5–2.0 lít/ha Kasumin hoặc 1.5 kg/ha Bactericide",
                "tổng_cho_1.8ha": "2.7–3.6 lít Kasumin",
                "pha_bình_25l": "40–50ml Kasumin"
            },
            "additional_actions_right_now": [
                "Ngưng bón đạm ngay lập tức",
                "Tháo nước ruộng xuống 5–8cm, giữ ruộng khô ráo 3–4 ngày",
                "Phun 2 đợt cách nhau 4–5 ngày"
            ]
        }
        }
    },
    {
        "farmer_id": "FARMER073",
        "full_name": "Trần Văn Út",
        "farm_name": "Ruộng phèn U Minh Thượng",
        "area_ha": 1.4,
        "planting_date": "2025-08-08",
        "days_since_planting": 92,
        "location": { "province": "Kiên Giang" },
        "iot_data": {
            "temperature": 29.2,
            "humidity": 79,
            "soil_moisture": 62,
            "soil_ph": 4.5,
            "water_level": 6,
            "lux": 52000,
            "wind": 3.1,
            "wind_avg": 2.8,
            "detected_disease_name": "brown_spot",
            "disease_confidence": 0.91
        },
        "summary_3d": {
            "20/11/2025": { "temperature": 29.5, "humidity": 77, "soil_moisture": 60, "water_level": 5, "lux": 55000, "wind": 3.5 },
            "21/11/2025": { "temperature": 29.0, "humidity": 80, "soil_moisture": 63, "water_level": 7, "lux": 50000, "wind": 2.9 },
            "22/11/2025": { "temperature": 28.8, "humidity": 81, "soil_moisture": 64, "water_level": 8, "lux": 48000, "wind": 2.6 }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "CAO",
            "treatment_plan": {
            "is_actionable": True,
            "main_message": "BỔ SUNG KALI + SILIC KHẨN CẤP KẾT HỢP PHUN THUỐC TRỊ ĐỐM NÂU",
            "optimal_spray_day": {
                "date": "2025-11-21",
                "session": "Buổi sáng",
                "reason": "Ngày có gió tốt nhất trong 3 ngày tới, giúp thuốc bám đều và khô nhanh trên lá."
            },
            "drug_info": {
                "sản_phẩm_khuyến_cáo": "Amistar Top 325SC hoặc Nativo 75WG",
                "hoạt_chất": "Azoxystrobin 200g/L + Difenoconazole 125g/L",
                "liều_lượng": "0.8–1.0 lít/ha Amistar Top",
                "tổng_cho_1.4ha": "1.12–1.4 lít",
                "pha_bình_25l": "20–25ml Amistar Top"
            },
            "fertilizer_advice": {
                "immediate": "Bón ngay hôm nay: 30kg KCl + 20kg Silic Canxi (SiO₂ 25%) cho 1.4ha",
                "next_3days": "Phun thêm phân bón lá chứa Bo + Zn + Silic (ví dụ: Siêu Bo, Siêu Silic)"
            },
            "additional_actions_right_now": [
                "Rút nước ruộng xuống còn 3–5cm để giảm ẩm đất phèn",
                "Phun thuốc chiều 21/11 và nhắc lại sau 7 ngày"
            ]
            }
        }
    },
    {
        "farmer_id": "FARMER089",
        "full_name": "Nguyễn Thị Kim Liên",
        "farm_name": "Ruộng Thoại Sơn",
        "area_ha": 2.2,
        "planting_date": "2025-08-18",
        "days_since_planting": 87,
        "location": { "province": "An Giang" },
        "iot_data": {
            "temperature": 27.8,
            "humidity": 91,
            "soil_moisture": 88,
            "soil_ph": 5.7,
            "water_level": 22,
            "lux": 28000,
            "wind": 4.8,
            "wind_avg": 4.5,
            "detected_disease_name": "bacterial_leaf_blight",
            "disease_confidence": 0.79
        },
        "summary_3d": {
            "25/11/2025": { "temperature": 27.5, "humidity": 93, "soil_moisture": 90, "water_level": 25, "lux": 55000, "wind": 5.2 },
            "26/11/2025": { "temperature": 28.0, "humidity": 89, "soil_moisture": 87, "water_level": 20, "lux": 56000, "wind": 4.1 },
            "27/11/2025": { "temperature": 28.2, "humidity": 87, "soil_moisture": 85, "water_level": 18, "lux": 47000, "wind": 3.8 }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "KHẨN CẤP",
            "treatment_plan": {
            "is_actionable": True,
            "main_message": "PHUN ĐỒNG + KASUGAMYCIN NGAY, GIỮ RUỘNG KHÔ",
            "optimal_spray_day": {
                "date": "2025-11-27",
                "session": "Buổi sáng",
                "reason": "Ngày duy nhất gió giảm nhẹ và trời tạnh ráo tương đối, thuốc đồng khô nhanh, không bị rửa trôi."
            },
            "drug_info": {
                "sản_phẩm_khuyến_cáo": "Kasumin 47WP + Copper B",
                "hoạt_chất": "Kasugamycin 47% + Đồng oxychloride",
                "liều_lượng": "1.5 kg Kasumin + 1.5 kg Copper B /ha",
                "tổng_cho_2.2ha": "3.3kg Kasumin + 3.3kg Copper B",
                "pha_bình_25l": "40g Kasumin + 40g Copper B"
            },
            "additional_actions_right_now": [
                "Tháo cạn nước ngay hôm nay, chỉ để 5–8cm",
                "Ngưng bón tất cả các loại đạm",
                "Phun 2 đợt cách nhau 5 ngày (26/11 và 1/12)"
            ]
            }
        }
    },
    {
        "farmer_id": "FARMER134",
        "full_name": "Võ Thị Ngọc Hân",
        "farm_name": "Ruộng Tân Hưng nhiễm nặng",
        "area_ha": 2.8,
        "planting_date": "2025-08-12",
        "days_since_planting": 94,
        "location": { "province": "Long An" },
        "iot_data": {
            "temperature": 26.5,
            "humidity": 95,
            "soil_moisture": 92,
            "soil_ph": 5.6,
            "water_level": 28,
            "lux": 18000,
            "wind": 1.2,
            "wind_avg": 1.0,
            "detected_disease_name": "bacterial_leaf_blight",
            "disease_confidence": 0.94
        },
        "summary_3d": {
            "1/12/2025": { "temperature": 26.2, "humidity": 96, "soil_moisture": 90,"water_level": 30, "lux": 41000, "wind": 0.8 },
            "2/12/2025": { "temperature": 26.8, "humidity": 92, "soil_moisture": 40,"water_level": 24, "lux": 47000, "wind": 2.1 },
            "3/12/2025": { "temperature": 27.5, "humidity": 88, "soil_moisture": 70,"water_level": 20, "lux": 50000, "wind": 3.5 }
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "CỰC KỲ KHẨN CẤP",
            "treatment_plan": {
            "is_actionable": True,
            "main_message": "CẮT NƯỚC KHẨN CẤP + PHUN LIÊN TỤC 3 ĐỢT KASUGAMYCIN + ĐỒNG",
            "optimal_spray_day": {
                "date": "2025-12-02",
                "session": "Buổi sáng",
                "reason": "Ngày đầu tiên có gió >2m/s và độ ẩm giảm, giúp thuốc đồng phát huy tác dụng."
            },
            "drug_info": {
                "sản_phẩm_khuyến_cáo": "Kasumin 2L + Kocide 2000",
                "liều_lượng": "2.0 lít Kasumin + 2.0 kg Kocide /ha",
                "tổng_cho_2.8ha": "5.6 lít Kasumin + 5.6 kg Kocide",
                "lịch_phun": "Đợt 1: 2/12, đợt 2: 6/12, đợt 3: 10/12"
            }, 
            "additional_actions_right_now": [
                "Tháo khô ruộng ngay lập tức (chỉ để 3–5cm nước)",
                "Cắt toàn bộ lá bị cháy nặng dưới cổ bông",
                "Không bón bất kỳ loại phân nào trong 10 ngày tới"
            ]
            }
        }
    },
    {
        "farmer_id": "FARMER142",
        "full_name": "Lê Văn Bé",
        "farm_name": "Ruộng Lung Tre",
        "area_ha": 2.0,
        "planting_date": "2025-07-25",
        "days_since_planting": 100,
        "location": { "province": "Đồng Tháp" },
        "iot_data": {
            "temperature": 26.7,
            "humidity": 93,
            "soil_moisture": 91,
            "soil_ph": 4.9,
            "water_level": 26,
            "lux": 21000,
            "wind": 1.9,
            "wind_avg": 2.2,
            "detected_disease_name": "brown_spot",
            "disease_confidence": 0.91
        },
        "summary_3d": {
            "25/11/2025": { "temperature": 26.2, "humidity": 97, "soil_moisture": 94, "water_level": 30, "lux": 12000, "wind": 3.8 },
            "26/11/2025": { "temperature": 26.4, "humidity": 96, "soil_moisture": 93, "water_level": 28, "lux": 14000, "wind": 3.5 },
            "27/11/2025": { "temperature": 27.0, "humidity": 91, "soil_moisture": 89, "water_level": 24, "lux": 22000, "wind": 1.7 }  
        },
        "expect_plan": {
            "is_action_needed": True,
            "priority": "KHẨN CẤP",
            "treatment_plan": {
                "is_actionable": True,
                "main_message": "BÓN KALI KHẨN + PHUN TRICYCLAZOLE NGAY 27/11, GIỮ RUỘNG KHÔ",
                "optimal_spray_day": {
                    "date": "2025-11-27",
                    "session": "Buổi sáng/chiều",
                    "reason": "Ngày 27/11 gió nhẹ nhất (1.7 m/s), độ ẩm giảm xuống 91%, ánh sáng tăng → điều kiện lý tưởng để phun Beam/Fuji-One: thuốc khô nhanh, bám lá tốt, ít bị rửa trôi, không bị phân hủy bởi nắng mạnh."
                },
                "drug_info": {
                    "sản_phẩm_khuyến_cáo": "Beam 75WP hoặc Kasai 75WP (Tricyclazole)",
                    "hoạt_chất": "Tricyclazole 75%",
                    "liều_lượng": "0.6–0.7 kg/ha",
                    "tổng_cho_2.0ha": "1.2–1.4 kg Beam 75WP",
                    "pha_bình_25l": "20g Beam 75WP"
                },
                "additional_actions_right_now": [
                    "Tháo nước ngay hôm nay (27/11), chỉ giữ 3–5 cm, để mặt ruộng khô ráo trước khi phun",
                    "Bón ngay 10–12 kg KCl/ha (tổng 20–24 kg cho 2 ha) vào chiều tối nay hoặc sáng mai",
                    "Ngưng bón mọi loại đạm",
                    "Phun đợt 1: 28/11, đợt 2: 04–05/12/2025",
                    "Nếu 2–3 ngày tới trời vẫn âm u kéo dài, chuẩn bị thêm 1 bình thuốc dự phòng"
                ]
            }
        }
    }
]

In [27]:
total = {
    "action": 0,
    "grounding_enough": 0,
    "risk": 0,
    "spray_date": 0,
    "spray_session": 0,
    "dose": 0,
}
total_latency = 0
case_count = 0

for i, item in enumerate(sample_farmer_data):

    expected_plan = item.get('expect_plan')
    if not expected_plan:
        print("⚠️ Bỏ qua case không có expect_plan.")
        continue

    start = time.time()
    actual_plan = create_treatment_plan_from_sample(item, client)
    latency = time.time() - start

    report = evaluate_plan_vs_expect(actual_plan, expected_plan, latency)

    # Tích lũy
    total_latency += report["latency_s"]
    total["action"] += report["action_accuracy"]
    total["grounding_enough"] += report["grounding_enough"]
    total["risk"] += report["risk_assessment_accuracy"]
    total["spray_date"] += report["spray_date_accuracy"]
    total["spray_session"] += report["spray_session_accuracy"]
    total["dose"] += report["dose_quality_accuracy"]

    case_count += 1

Đang tải kho 'blast' từ cache...
Tải thành công kho 'blast' với 129 vector.
Đã truy xuất 20 đoạn văn bản từ kho 'blast' cho câu hỏi: 'bệnh Đạo ôn giai đoạn 44 ngày sau sạ hoạt chất đặc...'
Đang tải kho 'brown_spot' từ cache...
Tải thành công kho 'brown_spot' với 102 vector.
Đã truy xuất 20 đoạn văn bản từ kho 'brown_spot' cho câu hỏi: 'bệnh Đốm nâu giai đoạn 54 ngày sau sạ hoạt chất đặ...'
Đang tải kho 'bacterial_leaf_blight' từ cache...
Tải thành công kho 'bacterial_leaf_blight' với 134 vector.
Đã truy xuất 20 đoạn văn bản từ kho 'bacterial_leaf_blight' cho câu hỏi: 'bệnh Cháy bìa lá giai đoạn 33 ngày sau sạ hoạt chấ...'
Đã truy xuất 20 đoạn văn bản từ kho 'blast' cho câu hỏi: 'bệnh Đạo ôn giai đoạn 80 ngày sau sạ hoạt chất đặc...'
Đã truy xuất 20 đoạn văn bản từ kho 'blast' cho câu hỏi: 'bệnh Đạo ôn giai đoạn 85 ngày sau sạ hoạt chất đặc...'
Đã truy xuất 20 đoạn văn bản từ kho 'bacterial_leaf_blight' cho câu hỏi: 'bệnh Cháy bìa lá giai đoạn 82 ngày sau sạ hoạt chấ...'
Đã truy xuất 20

In [28]:
# ==== Tổng kết ====
print("\n==================== TỔNG KẾT ====================")
print(f"🧪 Tổng số case: {case_count}")
print(f"⚡ Latency TB: {total_latency / case_count:.2f}s\n")

print("🎯 Accuracy theo từng tiêu chí:")
print(f"- Action Decision: {total['action'] / case_count:.2f}")
print(f"- Grounding >=5: {total['grounding_enough'] / case_count:.2f}")
print(f"- Risk Assessment: {total['risk'] / case_count:.2f}")
print(f"- Spray Date: {total['spray_date'] / case_count:.2f}")
print(f"- Spray Session: {total['spray_session'] / case_count:.2f}")
print(f"- Dose Quality: {total['dose'] / case_count:.2f}")
print("=================================================")


==================== TỔNG KẾT ====================
🧪 Tổng số case: 10
⚡ Latency TB: 16.22s

🎯 Accuracy theo từng tiêu chí:
- Action Decision: 1.00
- Grounding >=5: 0.80
- Risk Assessment: 1.00
- Spray Date: 0.80
- Spray Session: 0.70
- Dose Quality: 0.80
